# XGBoost

In [1]:
import pandas as pd
import numpy as np

import xgboost as xgb
from sklearn.model_selection import GroupKFold
from sklearn.multioutput import MultiOutputClassifier 
from sklearn.metrics import f1_score

import re #mi serve per pulire


from pathlib import Path
import warnings
# Nascondo i warning
warnings.filterwarnings('ignore')

# Definisco il percorso dei file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

# Lista dei csv su cui fare training
datasets = {
    't2_medsam': FILE_PATH / 't2_medsam_masks.csv',
    't2_preprocessed': FILE_PATH / 't2_preprocessed_masks.csv',
    't2_original': FILE_PATH / 't2_original_masks.csv',
    'medsam_dynamic': FILE_PATH / 'medsam_dynamic.csv',
    'preprocessed_dynamic': FILE_PATH / 'preprocessed_dynamic.csv',
    'original_dynamic': FILE_PATH / 'original_dynamic.csv'
}

# Training


In [2]:
def training(file_path, csv_name):
    # Vado a leggere il csv
    df = pd.read_csv(file_path)

    # Definisco le colonne target
    original_target_list = ['PR [SII]', 'ER [SII]', 'KI67 [%]']
    
    # Vado a rimuovere le lesioni (righe) non valide
    df_validi = df.dropna(subset=original_target_list).copy()

    # Trasformo tutto in valori binari per "facilitare" il lavoro
    df_validi['PR_class'] = (df_validi['PR [SII]'] > 0.5).astype(int)
    df_validi['ER_class'] = (df_validi['ER [SII]'] > 0.5).astype(int)
    df_validi['KI67_class'] = (df_validi['KI67 [%]'] >= 20).astype(int)  # Soglia clinica comune per KI67
    
    # Lista finale delle colonne target binarizzate che verranno usate per l'addestramento
    final_target_list = ['PR_class', 'ER_class', 'KI67_class']

    """
     Preparo le feature (X) e i target (y) per il modello
    """
    # Definisco tutte le colonne da rimuovere per ottenere solo le feature radiomiche
    features_to_drop = ['Patient ID', 'lesion idx', 'tumor/benign', 'GRADE', 'isTN', 'Breast'] + original_target_list + final_target_list
    features = df_validi.drop(columns=features_to_drop, errors='ignore')

    # 'target' contiene le 3 colonne da usare
    target = df_validi[final_target_list]
    
    # 'groups' contiene l'ID del paziente per ogni lesione.
    # Mi serve per fare la cross-validation a gruppo
    groups = df_validi['Patient ID']

    # Riempip a Nan se è rimasto vuoto
    features = features.fillna(features.mean())

    """ Dovrei pulire il nome delle colonne per farlo andare """
    features.columns = [re.sub(r'\[|\]|<', '', col) for col in features.columns]

    
    # Istanzio il classificatore XGBClassifier con parametri standard
    # random_state=42 garantisce che i risultati siano riproducibili
    xgb_clf = xgb.XGBClassifier(
        n_estimators=100,
        random_state=42,
        use_label_encoder=False,
        eval_metric='logloss' # Metrica per la valutazione interna del modello
    )
    
    # XGBoost non supporta la predizione multi-output
    multi_output_xgb = MultiOutputClassifier(xgb_clf)


    # Imposto la strategia di cross-validation.
    # GroupKFold assicura che le lesioni dello stesso paziente non vengano mai divise tra training set e test set
    cv = GroupKFold(n_splits=5, shuffle=True, random_state=42)
    
    # Lista vuota per collezionare i punteggi di performance di ogni fold.
    scores = []

    # Itero manualmente attraverso le 5 fold definite da GroupKFold.
    # 'enumerate' tiene traccia del numero della fold corrente.
    for fold, (train_index, test_index) in enumerate(cv.split(features, target, groups)):
        
        # Suddivide i dati in set di training e di test per la fold corrente.
        X_train, X_test = features.iloc[train_index], features.iloc[test_index]
        y_train, y_test = target.iloc[train_index], target.iloc[test_index]



        # ========== DEBUGGING: Stampo indici train/test  ==========

         
        """print("?"*50 + "\nDebug\n" + "?"*50)
        print(f"\nFold {fold} - File: {csv_name}")        
        print(f"  Train indice: {train_index[:10]})")
        print(f"  Test indice: {test_index[:10]})")
        print(f"  Train gruppo (Patient IDs): {groups.iloc[train_index].unique()}")
        print(f"  Test gruppo (Patient IDs): {groups.iloc[test_index].unique()}")
        print("?"*100)"""
        
        
        # ==========================================================            
        

        # Qui presumo che la fold sia valida, quindi inizio il treining
        multi_output_xgb.fit(X_train, y_train)
        y_pred = multi_output_xgb.predict(X_test)
        
        # Calcolo l'F1-score
        score = f1_score(y_test, y_pred, average='micro', zero_division=0)
        scores.append(score)

    # Converto la lista di punteggi in un array numpy per facilitare i calcoli.
    scores = np.array(scores)


    return {
        'mean_score': scores.mean(),      # Performance media sulle 5 fold
        'std_score': scores.std(),        # Variabilità della performance
        'scores_per_fold': scores         # Lista dei 5 punteggi individuali
    }

# Lettura dei file

In [3]:
results = {}
print("="*50 + "\n XGBoost\n" + "="*50)
for name, file_path in datasets.items():
    results[name] = training(file_path, name)

# Stampo i risultati 
for name, metrics in results.items():
    # Estraggo i 5 punteggi per il modello corrente
    scores_per_fold = metrics['scores_per_fold']
    
    # Formatto i punteggi in una stringa pulita
    formatted_scores = [f'{s:.3f}' for s in scores_per_fold]
    
    # Stampo la riga per il modello corrente
    print(f"\nNome CSV: {name}")
    #print(f"    scores per forld: {formatted_scores}")
    print(f"    Media e Dev. Std.: {metrics['mean_score']:.3f} ± {metrics['std_score']:.3f}")

 XGBoost

Nome CSV: t2_medsam
    Media e Dev. Std.: 0.707 ± 0.105

Nome CSV: t2_preprocessed
    Media e Dev. Std.: 0.723 ± 0.107

Nome CSV: t2_original
    Media e Dev. Std.: 0.718 ± 0.051

Nome CSV: medsam_dynamic
    Media e Dev. Std.: 0.688 ± 0.097

Nome CSV: preprocessed_dynamic
    Media e Dev. Std.: 0.724 ± 0.075

Nome CSV: original_dynamic
    Media e Dev. Std.: 0.709 ± 0.082
